In [1]:
import json
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from torch.utils.tensorboard import SummaryWriter


In [ ]:
def set_seed(seed: int) -> None:
    """What: Sets RNG seeds across Python, NumPy, and PyTorch.
    Purpose: Makes training runs reproducible.
    Needed for: Debugging and fair experiment comparison.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


INPUT_DIM = 64
OUTPUT_DIM = 128

CONFIG = {
    "seed": 42,
    "batch_size": 8,
    "num_epochs": 20,
    "learning_rate": 1e-3,
    "train_split": 0.9,
    "num_workers": 0,
    "log_interval": 10,
    "checkpoint_interval": 1,
    "run_root": Path("./runs"),
    "checkpoint_root": Path("./checkpoints"),
    "data_file": Path("./data/degraded_list.npy"),
    "gt_file": Path("./data/gt_list.npy"),
    "input_patch_shape": (INPUT_DIM, INPUT_DIM, INPUT_DIM),
    "output_patch_shape": (OUTPUT_DIM, OUTPUT_DIM, OUTPUT_DIM),
    "samples_per_timepoint": 2,
    "resume_checkpoint": None,
}

if any(o <= i for i, o in zip(CONFIG["input_patch_shape"], CONFIG["output_patch_shape"])):
    raise ValueError("For super-resolution, output_patch_shape must be larger than input_patch_shape in every dimension")

set_seed(CONFIG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used: {device}")

Device used: cuda


In [3]:
class SRCNN3D(nn.Module):
    def __init__(self, output_patch_shape=(50, 50, 50)):
        """What: Builds a 3D SRCNN-style network for SR.
        Purpose: Learn mapping from low-res patches to high-res patches.
        Needed for: Core model used in training and inference.
        """
        super(SRCNN3D, self).__init__()
        self.output_patch_shape = tuple(output_patch_shape)

        # Keep spatial shape after upsampling by using padding.
        self.conv1 = nn.Conv3d(in_channels=1, out_channels=64, kernel_size=9, stride=1, padding=4)
        self.relu1 = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv3d(in_channels=64, out_channels=32, kernel_size=1, stride=1, padding=0)
        self.relu2 = nn.ReLU(inplace=True)

        self.conv3 = nn.Conv3d(in_channels=32, out_channels=1, kernel_size=5, stride=1, padding=2)

        self._initialize_weights()

    def forward(self, x):
        """What: Upsamples input and refines it with 3D convolutions.
        Purpose: Produce super-resolved output volume.
        Needed for: Computing predictions and training loss.
        """
        x = F.interpolate(x, size=self.output_patch_shape, mode="trilinear", align_corners=False)
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.conv3(x)
        return x

    def _initialize_weights(self):
        """What: Initializes convolution weights and biases.
        Purpose: Start optimization from stable initial parameters.
        Needed for: Better convergence behavior at training start.
        """
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.normal_(m.weight, mean=0.0, std=0.001)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

In [4]:
# Initialize the model
model = SRCNN3D(output_patch_shape=CONFIG["output_patch_shape"]).to(device)

# Super-resolution: output must be larger than input.
dummy_input = torch.randn(1, 1, INPUT_DIM, INPUT_DIM, INPUT_DIM, device=device)
with torch.no_grad():
    output = model(dummy_input)

print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")
assert output.shape[-3:] > dummy_input.shape[-3:], "Output must be larger than input"

Input shape: torch.Size([1, 1, 33, 33, 33])
Output shape: torch.Size([1, 1, 50, 50, 50])


In [5]:
def center_crop_3d(volume: np.ndarray, target_shape: tuple[int, int, int]) -> np.ndarray:
    """What: Extracts the centered sub-volume with a given shape.
    Purpose: Spatially align targets without changing intensity values.
    Needed for: Tasks where model output should match a center ROI.
    """
    if volume.ndim != 3:
        raise ValueError(f"Expected 3D volume, got shape {volume.shape}")
    d, h, w = volume.shape
    td, th, tw = target_shape
    if d < td or h < th or w < tw:
        raise ValueError(f"Cannot crop volume {volume.shape} to {target_shape}")

    d0 = (d - td) // 2
    h0 = (h - th) // 2
    w0 = (w - tw) // 2
    return volume[d0:d0 + td, h0:h0 + th, w0:w0 + tw]


def normalize_minmax(arr: np.ndarray) -> np.ndarray:
    """What: Scales array values into [0, 1].
    Purpose: Normalize dynamic range across different samples.
    Needed for: Stable optimization and comparable loss scale.
    """
    arr = arr.astype(np.float32)
    amin = float(arr.min())
    amax = float(arr.max())
    if amax - amin < 1e-8:
        return np.zeros_like(arr, dtype=np.float32)
    return (arr - amin) / (amax - amin)


def resize_3d_numpy(volume: np.ndarray, target_shape: tuple[int, int, int]) -> np.ndarray:
    """What: Resamples a 3D NumPy volume to a target shape.
    Purpose: Create HR targets that match SR model output size.
    Needed for: Valid loss computation between output and label.
    """
    tensor = torch.from_numpy(volume.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    resized = F.interpolate(tensor, size=target_shape, mode="trilinear", align_corners=False)
    return resized.squeeze(0).squeeze(0).numpy()


class SRVolumeDataset(Dataset):
    """Dataset for paired degraded/gt 4D fMRI arrays saved as object npy lists."""

    def __init__(
        self,
        degraded_npy_path: Path,
        gt_npy_path: Path,
        input_patch_shape=(INPUT_DIM, INPUT_DIM, INPUT_DIM),
        output_patch_shape=(OUTPUT_DIM, OUTPUT_DIM, OUTPUT_DIM),
        samples_per_timepoint: int = 2,
        seed: int = 42,
    ):
        """What: Loads paired 4D volumes and builds patch sampling index.
        Purpose: Provide training pairs (LR patch, HR patch).
        Needed for: Feeding batches to the SR training loop.
        """
        self.degraded_path = Path(degraded_npy_path)
        self.gt_path = Path(gt_npy_path)
        if not self.degraded_path.exists():
            raise FileNotFoundError(f"Degraded data file not found: {self.degraded_path}")
        if not self.gt_path.exists():
            raise FileNotFoundError(f"GT data file not found: {self.gt_path}")

        self.input_patch_shape = tuple(input_patch_shape)
        self.output_patch_shape = tuple(output_patch_shape)
        self.samples_per_timepoint = int(samples_per_timepoint)
        self.seed = int(seed)

        degraded_raw = np.load(self.degraded_path, allow_pickle=True)
        gt_raw = np.load(self.gt_path, allow_pickle=True)

        self.degraded_list = self._to_volume_list(degraded_raw, "degraded")
        self.gt_list = self._to_volume_list(gt_raw, "gt")

        if len(self.degraded_list) != len(self.gt_list):
            raise ValueError(
                f"Mismatching list lengths: degraded={len(self.degraded_list)} gt={len(self.gt_list)}"
            )

        self.paired_4d = []
        self.sample_index = []

        for vol_idx, (deg_4d, gt_4d) in enumerate(zip(self.degraded_list, self.gt_list)):
            if deg_4d.shape != gt_4d.shape:
                raise ValueError(f"Volume {vol_idx}: degraded {deg_4d.shape} != gt {gt_4d.shape}")
            if deg_4d.ndim != 4:
                raise ValueError(f"Volume {vol_idx} must be 4D (X,Y,Z,T), got {deg_4d.shape}")

            x, y, z, t = deg_4d.shape
            inx, iny, inz = self.input_patch_shape
            if x < inx or y < iny or z < inz:
                raise ValueError(
                    f"Volume {vol_idx} too small for input patch {self.input_patch_shape}: {deg_4d.shape}"
                )

            self.paired_4d.append((deg_4d.astype(np.float32), gt_4d.astype(np.float32)))
            for t_idx in range(t):
                for _ in range(self.samples_per_timepoint):
                    self.sample_index.append((vol_idx, t_idx))

        if len(self.sample_index) == 0:
            raise ValueError("No training samples built from degraded/gt data")

    def _to_volume_list(self, raw, name: str):
        """What: Converts raw loaded object into a list of 4D arrays.
        Purpose: Support both in-memory arrays and path-based items.
        Needed for: Robust dataset loading from multiple storage formats.
        """
        if isinstance(raw, np.ndarray) and raw.dtype == object:
            values = raw.tolist()
        elif isinstance(raw, np.ndarray):
            if raw.ndim == 5:
                values = [raw[i] for i in range(raw.shape[0])]
            else:
                raise ValueError(f"{name} array has unsupported shape {raw.shape}")
        else:
            values = list(raw)

        parsed = []
        for idx, item in enumerate(values):
            if isinstance(item, (str, Path)):
                p = Path(item)
                if not p.exists():
                    raise FileNotFoundError(f"{name} volume path does not exist: {p}")
                arr = np.load(p, allow_pickle=True)
            else:
                arr = np.asarray(item)

            if arr.ndim != 4:
                raise ValueError(f"{name} volume {idx} must be 4D (X,Y,Z,T), got {arr.shape}")
            parsed.append(arr)

        return parsed

    def __len__(self):
        """What: Returns number of generated training samples.
        Purpose: Let DataLoader know dataset length.
        Needed for: Batching, epoch sizing, and split logic.
        """
        return len(self.sample_index)

    def __getitem__(self, idx):
        """What: Samples one normalized LR patch and matching HR target patch.
        Purpose: Produce one supervised training pair.
        Needed for: Per-batch input/label tensors during training.
        """
        vol_idx, t_idx = self.sample_index[idx]
        deg_4d, gt_4d = self.paired_4d[vol_idx]

        deg_3d = normalize_minmax(deg_4d[:, :, :, t_idx])
        gt_3d = normalize_minmax(gt_4d[:, :, :, t_idx])

        inx, iny, inz = self.input_patch_shape
        x, y, z = deg_3d.shape

        rng = np.random.default_rng(self.seed + idx)
        x0 = int(rng.integers(0, x - inx + 1))
        y0 = int(rng.integers(0, y - iny + 1))
        z0 = int(rng.integers(0, z - inz + 1))

        x_patch = deg_3d[x0:x0 + inx, y0:y0 + iny, z0:z0 + inz]
        y_patch_full = gt_3d[x0:x0 + inx, y0:y0 + iny, z0:z0 + inz]
        y_patch = resize_3d_numpy(y_patch_full, self.output_patch_shape)

        x_tensor = torch.from_numpy(x_patch).unsqueeze(0)
        y_tensor = torch.from_numpy(y_patch).unsqueeze(0)
        return x_tensor, y_tensor


def create_dataloaders(config):
    """What: Builds train/validation DataLoaders from config.
    Purpose: Centralize dataset creation and split behavior.
    Needed for: Reusable setup in training and sanity checks.
    """
    dataset = SRVolumeDataset(
        degraded_npy_path=config["data_file"],
        gt_npy_path=config["gt_file"],
        input_patch_shape=config["input_patch_shape"],
        output_patch_shape=config["output_patch_shape"],
        samples_per_timepoint=config["samples_per_timepoint"],
        seed=config["seed"],
    )

    train_size = max(1, int(len(dataset) * config["train_split"]))
    val_size = len(dataset) - train_size

    if val_size == 0 and len(dataset) > 1:
        val_size = 1
        train_size = len(dataset) - 1

    generator = torch.Generator().manual_seed(config["seed"])
    if val_size > 0:
        train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)
    else:
        train_dataset, val_dataset = dataset, None

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=config["num_workers"],
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = None
    if val_dataset is not None:
        val_loader = DataLoader(
            val_dataset,
            batch_size=config["batch_size"],
            shuffle=False,
            num_workers=config["num_workers"],
            pin_memory=torch.cuda.is_available(),
        )

    return train_loader, val_loader, len(dataset)


loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

In [ ]:
def psnr_from_mse(mse_value: float, data_range: float = 1.0) -> float:
    """What: Converts MSE to PSNR.
    Purpose: Provide an interpretable image-quality metric.
    Needed for: Monitoring SR quality during training/validation.
    """
    if mse_value <= 1e-12:
        return 99.0
    return 10.0 * np.log10((data_range ** 2) / mse_value)


def validate_one_epoch(model, val_loader, loss_fn, device):
    """What: Runs one full validation pass without gradients.
    Purpose: Measure generalization performance on held-out data.
    Needed for: LR scheduling and best-checkpoint selection.
    """
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            running_loss += loss.item()

    return running_loss / max(1, len(val_loader))


def train_one_epoch(epoch_index, model, train_loader, optimizer, loss_fn, device, tb_writer, log_interval=10):
    """What: Executes one optimization epoch on training data.
    Purpose: Update model parameters to minimize reconstruction loss.
    Needed for: Learning SR mapping from LR to HR patches.
    """
    model.train()
    running_loss = 0.0
    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        global_step = epoch_index * len(train_loader) + batch_idx
        tb_writer.add_scalar("batch/loss_train", loss.item(), global_step)

        if (batch_idx + 1) % log_interval == 0:
            print(f"Epoch {epoch_index + 1} Batch {batch_idx + 1}/{len(train_loader)} loss={loss.item():.6f}")

    return running_loss / max(1, len(train_loader))


def save_checkpoint(path: Path, epoch: int, model, optimizer, best_val_loss: float, config: dict):
    """What: Saves model, optimizer, and metadata to disk.
    Purpose: Preserve progress and allow later resume/evaluation.
    Needed for: Fault tolerance and best/final model export.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "config": {k: str(v) if isinstance(v, Path) else v for k, v in config.items()},
        },
        path,
    )


def maybe_resume_training(model, optimizer, checkpoint_path, device):
    """What: Restores model/optimizer from checkpoint if provided.
    Purpose: Continue interrupted experiments from last saved state.
    Needed for: Long trainings and iterative experimentation.
    """
    if checkpoint_path is None:
        return 0, float("inf")

    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = int(checkpoint["epoch"]) + 1
    best_val_loss = float(checkpoint.get("best_val_loss", float("inf")))
    return start_epoch, best_val_loss


def run_training(config):
    """What: Orchestrates full SR training loop with logging/checkpoints.
    Purpose: Train model end-to-end from config settings.
    Needed for: Producing usable super-resolution model weights.
    """
    run_name = datetime.now().strftime("%Y%m%d_%H%M%S") + f"_bs{config['batch_size']}_lr{config['learning_rate']}"
    run_dir = config["run_root"] / run_name
    ckpt_dir = config["checkpoint_root"] / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    with open(run_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump({k: str(v) if isinstance(v, Path) else v for k, v in config.items()}, f, indent=2)

    train_loader, val_loader, dataset_size = create_dataloaders(config)
    print(f"Dataset size: {dataset_size}, train batches: {len(train_loader)}")
    if val_loader is not None:
        print(f"Validation batches: {len(val_loader)}")

    writer = SummaryWriter(log_dir=str(run_dir / "tb"))
    writer.add_text("run/name", run_name)
    writer.add_text("run/device", device)
    writer.add_text("run/dataset", str(config["data_file"]))

    start_epoch, best_val_loss = maybe_resume_training(
        model,
        optimizer,
        config["resume_checkpoint"],
        device,
    )

    for epoch in range(start_epoch, config["num_epochs"]):
        train_loss = train_one_epoch(
            epoch,
            model,
            train_loader,
            optimizer,
            loss_fn,
            device,
            writer,
            log_interval=config["log_interval"],
        )

        val_loss = None
        if val_loader is not None:
            val_loss = validate_one_epoch(model, val_loader, loss_fn, device)
            scheduler.step(val_loss)
        else:
            scheduler.step(train_loss)

        current_lr = optimizer.param_groups[0]["lr"]
        writer.add_scalar("epoch/loss_train", train_loss, epoch)
        writer.add_scalar("epoch/lr", current_lr, epoch)
        train_psnr = psnr_from_mse(train_loss)
        writer.add_scalar("epoch/psnr_train", train_psnr, epoch)

        if val_loss is not None:
            writer.add_scalar("epoch/loss_val", val_loss, epoch)
            val_psnr = psnr_from_mse(val_loss)
            writer.add_scalar("epoch/psnr_val", val_psnr, epoch)
            print(
                f"Epoch {epoch + 1}/{config['num_epochs']} "
                f"train={train_loss:.6f} val={val_loss:.6f} "
                f"psnr_train={train_psnr:.2f} psnr_val={val_psnr:.2f} lr={current_lr:.2e}"
            )
        else:
            print(
                f"Epoch {epoch + 1}/{config['num_epochs']} "
                f"train={train_loss:.6f} psnr_train={train_psnr:.2f} lr={current_lr:.2e}"
            )

        if (epoch + 1) % config["checkpoint_interval"] == 0:
            save_checkpoint(ckpt_dir / f"epoch_{epoch + 1:03d}.pt", epoch, model, optimizer, best_val_loss, config)

        metric_for_best = val_loss if val_loss is not None else train_loss
        if metric_for_best < best_val_loss:
            best_val_loss = metric_for_best
            save_checkpoint(ckpt_dir / "best.pt", epoch, model, optimizer, best_val_loss, config)

    save_checkpoint(ckpt_dir / "final.pt", config["num_epochs"] - 1, model, optimizer, best_val_loss, config)
    writer.close()
    print(f"Training complete. Logs: {run_dir / 'tb'} | Checkpoints: {ckpt_dir}")

: 

In [ ]:
def run_sanity_checks(config):
    """What: Runs a one-batch forward/backward pass with assertions.
    Purpose: Catch shape/data/gradient issues early.
    Needed for: Fast validation before long training jobs.
    """
    train_loader, _, _ = create_dataloaders(config)
    inputs, labels = next(iter(train_loader))

    assert inputs.ndim == 5 and labels.ndim == 5, "Expected BCHWD tensors"
    assert inputs.shape[1] == 1 and labels.shape[1] == 1, "Expected single-channel volumes"

    model.train()
    inputs = inputs.to(device)
    labels = labels.to(device)

    optimizer.zero_grad(set_to_none=True)
    outputs = model(inputs)
    assert outputs.shape == labels.shape, f"Output {outputs.shape} and labels {labels.shape} mismatch"

    loss = loss_fn(outputs, labels)
    loss.backward()
    optimizer.step()

    print(f"Sanity check passed. One-batch loss: {loss.item():.6f}")


def run_tiny_overfit_check(config, steps: int = 20):
    """What: Trains a tiny model on one sample for a few steps.
    Purpose: Verify the setup can overfit simple data.
    Needed for: Diagnosing training pipeline bugs quickly.
    """
    train_loader, _, _ = create_dataloaders(config)
    inputs, labels = next(iter(train_loader))
    inputs = inputs[:1].to(device)
    labels = labels[:1].to(device)

    tiny_model = SRCNN3D(output_patch_shape=config["output_patch_shape"]).to(device)
    tiny_optimizer = torch.optim.Adam(tiny_model.parameters(), lr=config["learning_rate"])

    losses = []
    tiny_model.train()
    for _ in range(steps):
        tiny_optimizer.zero_grad(set_to_none=True)
        outputs = tiny_model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        tiny_optimizer.step()
        losses.append(loss.item())

    print(f"Tiny overfit check: start={losses[0]:.6f}, end={losses[-1]:.6f}")
    if losses[-1] >= losses[0]:
        print("Warning: loss did not decrease in tiny overfit test. Inspect data normalization and target pairing.")


# Run this block when degraded_list.npy is prepared.L
run_sanity_checks(CONFIG)
run_tiny_overfit_check(CONFIG, steps=20)
#run_training(CONFIG)

## Training launch checklist

- Confirm `CONFIG["data_file"]` points to your prepared `degraded_list.npy`.
- Verify each sample provides paired degraded/target volumes.
- Start TensorBoard from project root: `tensorboard --logdir runs`.
- Run cells top-to-bottom once to ensure imports/model definitions are fresh.
- Run sanity and tiny-overfit checks before long training.
- For resume, set `CONFIG["resume_checkpoint"]` to a saved `.pt` checkpoint path.